In [2]:

import sys
sys.path.append('./BIRM/Bayesian-Invariant-Risk-Minmization')
from utils import *


import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix
from model import MLP as BaseMLP, EBD
from torch import optim, autograd
torch.manual_seed(0)


from matplotlib import pyplot as plt

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Using device: {device}')

Using device: mps


In [4]:
FORCE_DOWNLOAD = False
CACHE_FILE     = 'abide_connectivity_ho.csv'
PIPELINE       = 'cpac'
STRATEGY       = 'filt_noglobal'   # no global signal regression — standard for ASD
ATLAS          = 'rois_ho'         # Harvard-Oxford, 111 ROIs → 6105 FC features
S3_BASE        = (
    'https://s3.amazonaws.com/fcp-indi/data/Projects/'
    'ABIDE_Initiative/Outputs/{pipeline}/{strategy}/{atlas}/'
    '{file_id}_{atlas}.1D'
)
TIMEOUT        = 25
SLEEP          = 0.05
MIN_TIMEPOINTS = 100   # drop subjects with very short scans

# ── Parser ─────────────────────────────────────────────────────────────────────
def parse_1D_to_fc(text: str, file_id: str):
    """Parse .1D ROI time series → upper-triangle Fisher-z FC feature vector."""
    lines = [l.strip() for l in text.splitlines()
             if l.strip() and not l.startswith('#')]
    if not lines:
        return None
    try:
        ts = np.array([[float(x) for x in l.split()] for l in lines])
    except ValueError:
        return None
    # ts shape: (n_timepoints, n_rois)
    if ts.shape[0] < MIN_TIMEPOINTS:
        print(f'    SKIP {file_id}: only {ts.shape[0]} timepoints')
        return None
    # Pearson correlation + Fisher z-transform
    corr = np.corrcoef(ts.T)
    corr = np.clip(corr, -0.9999, 0.9999)
    z    = np.arctanh(corr)
    # Upper triangle indices (excluding diagonal)
    n   = corr.shape[0]
    idx = np.triu_indices(n, k=1)
    features = {f'fc_{i}_{j}': z[i, j] for i, j in zip(idx[0], idx[1])}
    return features

# ── Download ───────────────────────────────────────────────────────────────────
if FORCE_DOWNLOAD :
    # Load phenotypic file — reuse the one already in memory if available,
    # otherwise download it fresh
    try:
        pheno_fc = pheno.copy()
        print(f'Using existing pheno table ({len(pheno_fc)} subjects)')
    except NameError:
        pheno_url = ('https://s3.amazonaws.com/fcp-indi/data/Projects/'
                     'ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv')
        pheno_fc = pd.read_csv(pheno_url)
        print(f'Downloaded phenotypic file ({len(pheno_fc)} subjects)')

    rows, skipped = [], []

    for idx, row in pheno_fc.iterrows():
        file_id = str(row.get('FILE_ID', '')).strip()
        sub_id  = row.get('SUB_ID', idx)

        if not file_id or file_id.lower() == 'no_filename':
            skipped.append(sub_id)
            continue

        url = S3_BASE.format(
            pipeline=PIPELINE, strategy=STRATEGY,
            atlas=ATLAS, file_id=file_id)

        try:
            r = requests.get(url, timeout=TIMEOUT)
            if r.status_code != 200:
                raise IOError(f'HTTP {r.status_code}')
            features = parse_1D_to_fc(r.text, file_id)
            if features is None:
                skipped.append(sub_id)
                continue
            features['SUB_ID']   = sub_id
            features['FILE_ID']  = file_id
            features['SITE_ID']  = row.get('SITE_ID',  '')
            features['DX_GROUP'] = row.get('DX_GROUP', np.nan)  # 1=ASD, 2=Control
            features['AGE_AT_SCAN'] = row.get('AGE_AT_SCAN', np.nan)
            features['SEX']      = row.get('SEX',      np.nan)
            rows.append(features)

        except Exception as e:
            skipped.append(sub_id)
            print(f'  [{idx+1:4d}] SKIP {file_id}  ({e})')

        if (len(rows) + len(skipped)) % 50 == 0:
            print(f'  {len(rows)+len(skipped):4d} / {len(pheno_fc)} processed  '
                  f'| ok={len(rows)}  skip={len(skipped)}')
        time.sleep(SLEEP)

    fc_df = pd.DataFrame(rows)
    # Map DX_GROUP (1=ASD, 2=Control) to DX_LABEL
    fc_df['DX_LABEL'] = fc_df['DX_GROUP'].map({1: 'ASD', 2: 'Control'})
    fc_df.to_csv(CACHE_FILE, index=False)
    print(f'\nDone — {len(rows)} subjects saved to {CACHE_FILE} | {len(skipped)} skipped')

else:
    fc_df = pd.read_csv(CACHE_FILE)
    print(f'Loaded from cache: {CACHE_FILE}  ({len(fc_df)} rows)')

# ── Quick sanity check ─────────────────────────────────────────────────────────
fc_cols = [c for c in fc_df.columns if c.startswith('fc_')]
print(f'\nConnectivity table shape: {fc_df.shape}')
print(f'FC features: {len(fc_cols)}  (expected 6105 for 111-ROI HO atlas)')
print(f'Sites: {fc_df["SITE_ID"].nunique()}')
print(f'Diagnosis:\n{fc_df["DX_LABEL"].value_counts()}')
print(f'NaN in FC features: {fc_df[fc_cols].isna().sum().sum()}')

Loaded from cache: abide_connectivity_ho.csv  (1009 rows)

Connectivity table shape: (1009, 6112)
FC features: 6105  (expected 6105 for 111-ROI HO atlas)
Sites: 19
Diagnosis:
DX_LABEL
Control    516
ASD        493
Name: count, dtype: int64
NaN in FC features: 46106


In [5]:
number_of_regions = 111
connectivity_features = [f'fc_{i}_{j}' for i in range(number_of_regions) for j in range(i+1, number_of_regions)] 

connectivity_df = fc_df[connectivity_features + ['DX_LABEL', 'SITE_ID']]


In [6]:
thickness_raw = df = pd.read_csv('abide_thickness_raw.csv')

In [7]:
thickness_ids = set(thickness_raw['SUB_ID'].astype(str))
connectivity_ids = set(fc_df['SUB_ID'].astype(str))

both = thickness_ids & connectivity_ids
only_thickness = thickness_ids - connectivity_ids
only_connectivity = connectivity_ids - thickness_ids

print(f"Thickness only:    {len(thickness_raw)} subjects")
print(f"Connectivity only: {len(fc_df)} subjects")
print(f"In both:           {len(both)} subjects")
print(f"Only in thickness: {len(only_thickness)}")
print(f"Only in FC:        {len(only_connectivity)}")

Thickness only:    1035 subjects
Connectivity only: 1009 subjects
In both:           1009 subjects
Only in thickness: 26
Only in FC:        0


In [8]:
df = connectivity_df
print(f'Initial shape: {df.shape}')
df = df.dropna(axis=0)
df = df.reset_index(drop=True) 
print(f'After dropna: {df.shape}')

env_col = 'SITE_ID'
label_col = 'DX_LABEL'

feature_cols = [c for c in df.columns if c.startswith('fc_')]
df[label_col] = df[label_col].map({'Control': 0, 'ASD': 1}) 




Initial shape: (1009, 6107)
After dropna: (761, 6107)


In [9]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super(TabularMLP, self).__init__()
        self._main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GroupNorm(8, hidden_dim),
            nn.PReLU(),
            nn.Dropout(0.5),  
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GroupNorm(8, hidden_dim // 2),
            nn.PReLU(),
            nn.Dropout(0.5),  
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x):
        return self._main(x)


In [10]:
class TabularDataProvider(IRMDataProvider):
    def __init__(self, df, env_col, label_col, feature_cols, test_envs, device):
        super(TabularDataProvider, self).__init__(None)
        self.device = device
        train_mask = df[env_col].isin([e for e in df[env_col].unique() if e not in test_envs])
        scaler = StandardScaler()
        scaler.fit(df.loc[train_mask, feature_cols].values)
        features = scaler.transform(df[feature_cols].values)
        
        envs_unique = sorted(df[env_col].unique())
        self.envs = []
        for idx, env in enumerate(envs_unique):
            env_data = df[df[env_col] == env]
            env_features = torch.tensor(features[env_data.index], dtype=torch.float32).to(device)
            env_labels = torch.tensor(env_data[label_col].values, dtype=torch.float32).unsqueeze(1).to(device)
            env_g = torch.full((len(env_data),), idx, dtype=torch.long).to(device)
            env_c = torch.zeros(len(env_data), 1).to(device)
            self.envs.append({
                'images': env_features,
                'labels': env_labels,
                'color': env_c
            })
        # Reorder so test envs are last
        test_env_indices = [envs_unique.index(env) for env in test_envs if env in envs_unique]
        train_envs = [self.envs[i] for i in range(len(self.envs)) if i not in test_env_indices]
        test_envs_list = [self.envs[i] for i in test_env_indices]
        self.envs = train_envs + test_envs_list
        self.num_test = len(test_env_indices)
        self.preprocess_data()

    def preprocess_data(self):
        self.train_x, self.train_y, self.train_g, self.train_c = self.concat_envs_device(self.envs[:-self.num_test])
        self.test_x, self.test_y, self.test_g, self.test_c = self.concat_envs_device(self.envs[-self.num_test:])

    def concat_envs_device(self, con_envs):
        con_x = torch.cat([env["images"] for env in con_envs])
        con_y = torch.cat([env["labels"] for env in con_envs])
        con_g = torch.cat([
            ig * torch.ones_like(env["labels"])
            for ig,env in enumerate(con_envs)])
        con_c = torch.cat([env["color"] for env in con_envs])
        return con_x, con_y, con_g, con_c

In [11]:
class CustomEBD(nn.Module):
    def __init__(self, flags, device):
        super(CustomEBD, self).__init__()
        self.flags = flags
        self.device = device
        self.embedings = torch.nn.Embedding(flags.envs_num, 1).to(device)
        self.re_init()

    def re_init(self):
        self.embedings.weight.data.fill_(1.)

    def re_init_with_noise(self, noise_sd):
        rd = torch.normal(
            torch.Tensor([1.0] * self.flags.envs_num),
            torch.Tensor([noise_sd] * self.flags.envs_num))
        self.embedings.weight.data = rd.view(-1, 1).to(self.device)

    def forward(self, e):
        return self.embedings(e.long())

In [12]:
test_envs = ['UCLA_1',
             'UM_1'
             ]
dp = TabularDataProvider(df, env_col, label_col, feature_cols, test_envs, device)
input_dim = len(feature_cols)
mlp = TabularMLP(input_dim).to(device)
ebd = CustomEBD(type('Flags', (), {'envs_num': len(dp.envs)-len(test_envs)})(), device)

In [13]:
def train_model_fixed(mlp, ebd, dp, irm_type='birm',
                      steps=2000, lr=4e-4,
                      penalty_weight=1e4,
                      penalty_anneal_iters=200,
                      sampleN=15,
                      prior_sd_coef=1.0,
                      verbose=True,
                      track_history=True,
                      weight_decay=1e-1):

    optimizer = optim.Adam(mlp.parameters(), lr=lr, weight_decay=weight_decay)
    loss_history = {'total': [], 'nll': [], 'penalty': []}
    acc_history  = {'train': [], 'test': []}
    n_train_envs = len(dp.envs) - dp.num_test

    for step in range(steps):
        train_x, train_y, train_g, train_c = dp.fetch_train()
        train_logits = mlp(train_x)

        if irm_type == 'birm':
            ebd.re_init()
            clean_logits = ebd(train_g).view(-1, 1) * train_logits
            clean_nll = nn.functional.binary_cross_entropy_with_logits(clean_logits, train_y)
            train_penalty = torch.tensor(0.0, device=device)
            for _ in range(sampleN):
                ebd.re_init_with_noise(prior_sd_coef)
                train_logits_w = ebd(train_g).view(-1, 1) * train_logits
                noisy_nll = nn.functional.binary_cross_entropy_with_logits(train_logits_w, train_y)
                grad = autograd.grad(noisy_nll * n_train_envs,
                                     ebd.parameters(), create_graph=True)[0]
                train_penalty = train_penalty + (1 / sampleN) * torch.mean(grad ** 2)
            w = min(penalty_weight, (step / max(penalty_anneal_iters, 1)) ** 2 * penalty_weight)
            loss = clean_nll + w * train_penalty
            if step == penalty_anneal_iters and verbose:
                print(f"  *** Penalty active: nll={clean_nll.item():.4f}, "
                      f"nll_vs_random={clean_nll.item()/0.693:.3f}")

        elif irm_type == 'erm':
            clean_nll = nn.functional.binary_cross_entropy_with_logits(train_logits, train_y)
            train_penalty = torch.tensor(0.0, device=device)
            loss = clean_nll

        elif irm_type == 'vrex':
            env_losses = []
            for g in range(n_train_envs):
                env_mask = (train_g == g).squeeze()
                if env_mask.sum() == 0:
                    continue
                env_loss = nn.functional.binary_cross_entropy_with_logits(
                    train_logits[env_mask], train_y[env_mask])
                env_losses.append(env_loss)
            env_losses = torch.stack(env_losses)
            clean_nll = env_losses.mean()
            train_penalty = env_losses.var(unbiased=False)
            ramp = min(1.0, (step / max(penalty_anneal_iters, 1)) ** 2)
            loss = clean_nll + penalty_weight * ramp * train_penalty

        else:
            raise ValueError(f"Unknown irm_type {irm_type}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if verbose and step % 200 == 0:
            print(f"Step {step}: loss={loss.item():.4f}, "
                  f"nll={clean_nll.item():.4f}, "
                  f"penalty={train_penalty.item():.6f}")

        if track_history and step % 200 == 0:
            loss_history['total'].append(loss.item())
            loss_history['nll'].append(clean_nll.item())
            loss_history['penalty'].append(train_penalty.item())
            train_acc, _, _, _ = evaluate_model(mlp, ebd, dp, eval_on='train')
            test_acc,  _, _, _ = evaluate_model(mlp, ebd, dp, eval_on='test')
            acc_history['train'].append(train_acc)
            acc_history['test'].append(test_acc)

    return mlp, ebd, loss_history, acc_history

In [14]:
def evaluate_model(mlp, ebd, dp, irm_type='birm', eval_on='test'):
    if eval_on == 'test':
        test_x, test_y, test_g, test_c = dp.fetch_test()
    else:
        test_x, test_y, test_g, test_c = dp.train_x, dp.train_y, dp.train_g, dp.train_c
    
    mlp.eval()
    with torch.no_grad():
        test_logits = mlp(test_x)
        
    preds = (torch.sigmoid(test_logits) > 0.5).float()
    acc = (preds == test_y).float().mean().item()
    probs = torch.sigmoid(test_logits).cpu().numpy()
    labels = test_y.cpu().numpy()
    
    auc = roc_auc_score(labels, probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds.cpu().numpy()).ravel()
    type1_error = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    type2_error = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    mlp.train()
    return acc, type1_error, type2_error, auc


First, testing vanila ERM:

In [ ]:

for seed in range(10):
    torch.manual_seed(seed)
    mlp_e = TabularMLP(input_dim).to(device)
    ebd_e = CustomEBD(type('Flags', (), {'envs_num': len(dp.envs)-len(test_envs)})(), device)
    mlp_e, ebd_e, losses_e, accs_e = train_model_fixed(mlp_e, None # no EBD for ERM
                                                       , dp, irm_type='erm',
                                        steps=3000, verbose=False, track_history=False)
    _, _, _, auc_e = evaluate_model(mlp_e, ebd_e, dp, irm_type='erm', eval_on='test')
    print(f'Seed {seed}: ERM test AUC = {auc_e:.4f}')


In [ ]:

for seed in range(10):
    torch.manual_seed(seed)
    mlp_v = TabularMLP(input_dim).to(device)
    ebd_v = CustomEBD(type('Flags', (), {'envs_num': len(dp.envs)-len(test_envs)})(), device)
    mlp_v, ebd_v, losses_v, accs_v = train_model_fixed(mlp_v, None # no EBD for VREX
                                                       , dp, irm_type='vrex',
                                        steps=3000, verbose=True, track_history=False, penalty_weight=10.0)
    _, _, _, auc_v = evaluate_model(mlp_v, ebd_v, dp, irm_type='vrex', eval_on='test')
    print(f'Seed {seed}: VREX test AUC = {auc_v:.4f}')

Step 0: loss=0.7102, nll=0.7102, penalty=0.004619
Step 200: loss=0.5659, nll=0.5209, penalty=0.004501
Step 400: loss=0.0617, nll=0.0615, penalty=0.000026
Step 600: loss=0.0917, nll=0.0868, penalty=0.000491
Step 800: loss=0.0563, nll=0.0561, penalty=0.000025
Step 1000: loss=0.0746, nll=0.0728, penalty=0.000187
Step 1200: loss=0.0709, nll=0.0698, penalty=0.000110
Step 1400: loss=0.0712, nll=0.0711, penalty=0.000017
Step 1600: loss=0.0902, nll=0.0888, penalty=0.000144
Step 1800: loss=0.0936, nll=0.0915, penalty=0.000210


KeyboardInterrupt: 

## trying feature selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

K = 500 
train_mask = ~df['SITE_ID'].isin(test_envs)

selector = SelectKBest(f_classif, k=K)
selector.fit(df.loc[train_mask, feature_cols].values,
             df.loc[train_mask, label_col].values)

selected_cols = [feature_cols[i] for i in selector.get_support(indices=True)]

feature_cols = selected_cols

In [ ]:
test_envs = ['UCLA_1',
             'UM_1'
             ]
dp = TabularDataProvider(df, env_col, label_col, feature_cols, test_envs, device)
input_dim = len(feature_cols)
mlp = TabularMLP(input_dim).to(device)
ebd = CustomEBD(type('Flags', (), {'envs_num': len(dp.envs)-len(test_envs)})(), device)

In [ ]:
configs = [
    {'hidden_dim': 32, 'weight_decay': 0.15},  
    {'hidden_dim': 64, 'weight_decay': 0.15}, 
    {'hidden_dim': 64, 'weight_decay': 0.2},   
]

for cfg in configs:
    aucs = []
    for seed in range(3):
        torch.manual_seed(seed)
        mlp_v = TabularMLP(input_dim, hidden_dim=cfg['hidden_dim']).to(device)
        mlp_v, _, _, _ = train_model_fixed(
            mlp_v, None, dp, irm_type='vrex',
            steps=2000, lr=1e-4,
            weight_decay=cfg['weight_decay'],
            penalty_weight=10.0,
            verbose=False, track_history=False)
        _, _, _, auc = evaluate_model(mlp_v, None, dp, eval_on='test')
        aucs.append(auc)
    print(f"hd={cfg['hidden_dim']}, wd={cfg['weight_decay']}: "
          f"mean={np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

hd=32, wd=0.15: mean=0.704 ± 0.019
hd=64, wd=0.15: mean=0.695 ± 0.005
hd=64, wd=0.2: mean=0.696 ± 0.012


In [ ]:
# ERM comparison with same regularization
for cfg in [{'hidden_dim': 32, 'weight_decay': 0.15},
            {'hidden_dim': 64, 'weight_decay': 0.15}]:
    aucs = []
    for seed in range(3):
        torch.manual_seed(seed)
        mlp_e = TabularMLP(input_dim, hidden_dim=cfg['hidden_dim']).to(device)
        mlp_e, _, _, _ = train_model_fixed(
            mlp_e, None, dp, irm_type='erm',
            steps=2000, lr=1e-4,
            weight_decay=cfg['weight_decay'],
            verbose=False, track_history=False)
        _, _, _, auc = evaluate_model(mlp_e, None, dp, eval_on='test')
        aucs.append(auc)
    print(f"ERM hd={cfg['hidden_dim']}, wd={cfg['weight_decay']}: "
          f"mean={np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

ERM hd=32, wd=0.15: mean=0.711 ± 0.011
ERM hd=64, wd=0.15: mean=0.674 ± 0.011


In [23]:
aucs_vrex, aucs_erm = [], []
for seed in range(10):
    torch.manual_seed(seed)
    
    mlp_v = TabularMLP(input_dim, hidden_dim=32).to(device)
    mlp_v, _, _, _ = train_model_fixed(
        mlp_v, None, dp, irm_type='vrex',
        steps=2000, lr=1e-4, weight_decay=0.15,
        penalty_weight=10.0, penalty_anneal_iters=0,
        verbose=False, track_history=False)
    _, _, _, auc = evaluate_model(mlp_v, None, dp, eval_on='test')
    aucs_vrex.append(auc)

    torch.manual_seed(seed)
    mlp_e = TabularMLP(input_dim, hidden_dim=64).to(device)
    mlp_e, _, _, _ = train_model_fixed(
        mlp_e, None, dp, irm_type='erm',
        steps=2000, lr=1e-4, weight_decay=0.15,
        verbose=False, track_history=False)
    _, _, _, auc = evaluate_model(mlp_e, None, dp, eval_on='test')
    aucs_erm.append(auc)

    print(f"Seed {seed}: VREx={aucs_vrex[-1]:.3f}  ERM={aucs_erm[-1]:.3f}  "
          f"Δ={aucs_vrex[-1]-aucs_erm[-1]:+.3f}")

print(f"\nVREx: {np.mean(aucs_vrex):.3f} ± {np.std(aucs_vrex):.3f}")
print(f"ERM:  {np.mean(aucs_erm):.3f} ± {np.std(aucs_erm):.3f}")
print(f"Mean gap: {np.mean(np.array(aucs_vrex)-np.array(aucs_erm)):+.3f}")

Seed 0: VREx=0.739  ERM=0.706  Δ=+0.034
Seed 1: VREx=0.756  ERM=0.717  Δ=+0.039
Seed 2: VREx=0.749  ERM=0.733  Δ=+0.016
Seed 3: VREx=0.743  ERM=0.746  Δ=-0.004
Seed 4: VREx=0.753  ERM=0.712  Δ=+0.041
Seed 5: VREx=0.752  ERM=0.731  Δ=+0.021
Seed 6: VREx=0.763  ERM=0.728  Δ=+0.035
Seed 7: VREx=0.760  ERM=0.734  Δ=+0.026
Seed 8: VREx=0.771  ERM=0.746  Δ=+0.025
Seed 9: VREx=0.767  ERM=0.716  Δ=+0.051

VREx: 0.755 ± 0.010
ERM:  0.727 ± 0.013
Mean gap: +0.028


In [ ]:
torch.manual_seed(8)
mlp_v = TabularMLP(input_dim, hidden_dim=32).to(device)
mlp_v, _, _, _ = train_model_fixed(
    mlp_v, None, dp, irm_type='vrex',
    steps=2000, lr=1e-4, weight_decay=0.15,
    penalty_weight=10.0, penalty_anneal_iters=0,
    verbose=True, track_history=False)

acc, t1, t2, auc = evaluate_model(mlp_v, None, dp, eval_on='test')
print(f"AUC={auc:.3f} | Acc={acc:.3f} ({acc*100:.1f}%) | T1={t1:.3f} | T2={t2:.3f}")

Step 0: loss=0.7274, nll=0.7274, penalty=0.004048
Step 200: loss=0.6779, nll=0.6642, penalty=0.001374
Step 400: loss=0.6499, nll=0.6390, penalty=0.001089
Step 600: loss=0.6175, nll=0.6035, penalty=0.001400
Step 800: loss=0.5865, nll=0.5682, penalty=0.001833
Step 1000: loss=0.5523, nll=0.5224, penalty=0.002997
Step 1200: loss=0.5257, nll=0.5059, penalty=0.001974
Step 1400: loss=0.4736, nll=0.4562, penalty=0.001742
Step 1600: loss=0.4242, nll=0.3991, penalty=0.002513
Step 1800: loss=0.3712, nll=0.3532, penalty=0.001802
AUC=0.771 | Acc=0.695 (69.5%) | T1=0.254 | T2=0.350


In [33]:
from scipy import stats
t, p = stats.ttest_rel(aucs_vrex, aucs_erm)
print(f"Paired t-test: t={t:.3f}, p={p:.5f}")
wins = sum(v > e for v, e in zip(aucs_vrex, aucs_erm))

Paired t-test: t=5.826, p=0.00025


In [27]:
t_ind, p_ind = stats.ttest_ind(aucs_vrex, aucs_erm)
print(f"Independent t-test: t={t_ind:.3f}, p={p_ind:.5f}")

Independent t-test: t=5.200, p=0.00006


BIRM

In [15]:

aucs_birm = []

for seed in range(10):
    torch.manual_seed(seed)
    mlp_b = TabularMLP(input_dim, hidden_dim=32).to(device)
    ebd_b = CustomEBD(
        type('Flags', (), {'envs_num': len(dp.envs) - len(test_envs)})(),
        device)

    mlp_b, ebd_b, _, _ = train_model_fixed(
        mlp_b, ebd_b, dp,
        irm_type='birm',
        steps=2000,
        lr=1e-4,
        weight_decay=0.15,
        penalty_weight=100,
        penalty_anneal_iters=500,
        sampleN=15,
        prior_sd_coef=0.5,
        verbose=(seed == 0),    
        track_history=False)

    _, _, _, auc = evaluate_model(mlp_b, None, dp, eval_on='test')
    aucs_birm.append(auc)
    print(f"Seed {seed}: BIRM={auc:.3f}  ")
        #   f"VREx={aucs_vrex[seed]:.3f}  "
        #   f"ERM={aucs_erm[seed]:.3f}  "
        #   f"Δ(BIRM-ERM)={auc-aucs_erm[seed]:+.3f}")

print(f"\nBIRM: {np.mean(aucs_birm):.3f} ± {np.std(aucs_birm):.3f}")

Step 0: loss=0.7411, nll=0.7411, penalty=0.013949
Step 200: loss=0.7292, nll=0.6701, penalty=0.003692
Step 400: loss=0.7464, nll=0.6609, penalty=0.001336
  *** Penalty active: nll=0.6796, nll_vs_random=0.981
Step 600: loss=0.7607, nll=0.6854, penalty=0.000753
Step 800: loss=0.7215, nll=0.6850, penalty=0.000365
Step 1000: loss=0.7092, nll=0.6892, penalty=0.000200
Step 1200: loss=0.7090, nll=0.6820, penalty=0.000270
Step 1400: loss=0.6968, nll=0.6868, penalty=0.000100
Step 1600: loss=0.6977, nll=0.6886, penalty=0.000091
Step 1800: loss=0.6946, nll=0.6879, penalty=0.000067
Seed 0: BIRM=0.555  
Seed 1: BIRM=0.487  
Seed 2: BIRM=0.603  
Seed 3: BIRM=0.559  
Seed 4: BIRM=0.580  
Seed 5: BIRM=0.593  
Seed 6: BIRM=0.584  
Seed 7: BIRM=0.564  
Seed 8: BIRM=0.582  
Seed 9: BIRM=0.522  

BIRM: 0.563 ± 0.033
